# 核心目标：

1. 机制（Mechanics）： 掌握 PyTorch 的基础操作（自底向上：张量 -> 构建模型 -> 优化器 -> 训练循环）。
2. 思维模式（Mindset）： 养成资源核算的习惯。你需要清楚每一行代码消耗了多少内存（GB）和算力（FLOPs）。
3. 直觉（Intuitions）： 理解为什么大模型需要特定的硬件和算法优化。

1. Tokenizer ：负责格式转换。它把人类的文字（字符串）切碎并转换成数字 ID 列表。它不具备任何“思考”或“理解”能力。
2. Transformer：负责语义理解和生成。它接收 Tokenizer 传来的数字序列，通过复杂的数学运算（注意力机制等）来预测下一个词。
3. Tokenizer 的核心：是字符串匹配和查表。涉及矩阵乘法，需要 GPU。
4. Transformer 的核心：是线性代数运算。它使用 Tensor（张量）、Attention（注意力层）和 MLP（多层感知机）。它必须在 GPU/NPU 上运行才能保证速度。
5. Transformer 的“知识”：存储在 Weights（权重参数）里。这些是通过深度学习训练出来的数亿个浮点数（如 7B, 67B 参数）
# AI 应用中看到的完整流程是这样的：
1. 输入："你好"
2. Tokenizer (你的类)：[1, 5, 20]（转为数字）
3. Embedding 层：将数字转为向量（矩阵）
4. Transformer 层：通过注意力机制处理这些向量
5. 输出层：预测下一个数字 ID [30]


In [1]:
import torch

In [2]:
x = torch.tensor(([1., 2., 3], [4, 5, 6]))
y = x[0]

In [3]:
y

tensor([1., 2., 3.])

In [4]:
y = x[:,1]
y

tensor([2., 5.])

In [5]:
y = x.view(3, 2)
y

tensor([[1., 2.],
        [3., 4.],
        [5., 6.]])

In [6]:
y = x.transpose(1, 0)

In [7]:
y

tensor([[1., 4.],
        [2., 5.],
        [3., 6.]])

In [8]:
assert not y.is_contiguous()

In [9]:
x1 = torch.zeros(4,8)
x1

tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [10]:
x2 = torch.randn(4, 8)
x2

tensor([[-3.6153e-01, -1.9758e+00,  6.1706e-01, -9.9822e-02,  1.6723e-02,
          1.7307e+00, -2.0728e-01,  1.7612e+00],
        [-2.6142e-01,  4.9878e-01, -8.6841e-01,  8.1093e-01,  1.0526e+00,
         -5.3409e-01, -7.0366e-01, -3.8472e-01],
        [-1.8324e-01,  1.2312e+00, -9.3201e-02, -7.6296e-04, -1.5726e-01,
          1.9844e+00,  3.0684e-01,  2.8673e-01],
        [-1.7231e-01, -1.7203e+00, -7.8028e-01,  1.5054e+00, -5.3188e-01,
          8.0340e-02,  1.1516e+00,  1.8354e-01]])

In [11]:
y

tensor([[1., 4.],
        [2., 5.],
        [3., 6.]])

In [12]:
y = x.transpose(1, 0).contiguous()
y

tensor([[1., 4.],
        [2., 5.],
        [3., 6.]])

In [13]:
y = x.transpose(1, 0).contiguous().view(2, 3)
y

tensor([[1., 4., 2.],
        [5., 3., 6.]])

#  PyTorch 处理内存的底层逻辑：逻辑视图与物理存储的分离。
1. 视图操作 (View-based / Narrowing Ops)
这些函数不拷贝数据。它们只是创建了一个新的 Tensor 对象，修改了“步长 (Stride)”或“偏移量 (Offset)”，但背后指向的是同一块内存地址。
常见函数：
view()
narrow()
expand()
transpose() / t() / permute()
squeeze() / unsqueeze()
slice (例如 x[0:2, :])
特点： 极其快（近乎瞬间完成），不额外占显存。
风险： 修改其中一个 Tensor，另一个也会跟着变（因为内存共享）
2. 拷贝操作 (Copy-based / Reallocation Ops)
这些函数会重新申请显存并物理搬运数据。
常见函数：
contiguous() (当 Tensor 不连续时)
clone()
to() (例如从 CPU 搬到 GPU)
detach().clone()
repeat() (注意与 expand 的区别，repeat 是物理复制)
特点： 消耗额外显存，有计算开销。

In [14]:
x = torch.tensor([1,4,9])
assert torch.equal(x.pow(2), torch.tensor([1, 16, 81]))

In [15]:
x1 = torch.tensor([1, 1,2])
assert torch.equal(x1.pow(2), torch.tensor([1,1,4]))

In [16]:
assert torch.equal(x.sqrt(), torch.tensor([1, 2, 3]))


In [17]:
assert torch.equal(x.rsqrt(), torch.tensor([1, 1/2, 1/3]))


In [18]:
x.pow(2) == torch.tensor([1, 16, 81])

tensor([True, True, True])

In [19]:
x = torch.ones(3, 3)
x

tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])

In [20]:
x = x.triu()
x

tensor([[1., 1., 1.],
        [0., 1., 1.],
        [0., 0., 1.]])

In [21]:
x = torch.ones(16, 32)
w = torch.ones(32, 2)
y = x @ w
assert y.size() == torch.Size([16, 2])

In [22]:
y.size()

torch.Size([16, 2])

In [23]:
a = torch.Size([16, 2])
a

torch.Size([16, 2])

In [24]:
x = torch.ones([2, 3,4])
x

tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]])

In [25]:
x = torch.ones([2, 2, 3, 4])

In [26]:
x

tensor([[[[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]],

         [[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]]],


        [[[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]],

         [[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]]]])

In [27]:
x1 = torch.ones(2, 2,3,4)
x1

tensor([[[[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]],

         [[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]]],


        [[[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]],

         [[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]]]])

In [28]:
assert torch.equal(x1, x)

In [29]:
x = torch.ones(2, 2,3)
y = torch.ones(2, 2, 3)
z = x @ y.transpose(-2, -1)
z.size()

torch.Size([2, 2, 2])

In [30]:
a = torch.ones(2, 3, 4)
a = a.transpose(1, 0)
a

tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.]]])

In [33]:
pip install jaxtyping


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.4 MB/s eta 0:00:00


In [34]:
pip install einops


In [35]:
x = torch.ones(2, 2, 1, 3)
from jaxtyping import Float

In [36]:
x:Float[torch.Tensor, "batch seq heads hidden"] = torch.ones(2, 2, 1, 3)

In [37]:
x

tensor([[[[1., 1., 1.]],

         [[1., 1., 1.]]],


        [[[1., 1., 1.]],

         [[1., 1., 1.]]]])

jaxtyping 提供的维度命名（如 "batch seq hidden"）在当前 PyTorch 生态中主要是“文档性质”的，不会在运行时自动强制检查维度是否真的匹配名称，但它能极大地提升代码的清晰度和可靠性。也就是说PyTorch 在运行时不会检查 x 的形状是否真的是 (batch=2, seq=3, hidden=4)
Python 的类型注解（Type Hints）本身是可选的、非强制的，主要用于工具（如 IDE、mypy）做静态分析，而不是运行时校验。尽管不强制校验

In [38]:
from einops import einsum

In [39]:
x: Float[torch.Tensor, "batch seq1 hidden"] = torch.ones(2, 3, 4)
y: Float[torch.Tensor, "batch seq2 hidden"] = torch.ones(2, 3, 4)


In [40]:
z = x @ y.transpose(-2, -1)
z

tensor([[[4., 4., 4.],
         [4., 4., 4.],
         [4., 4., 4.]],

        [[4., 4., 4.],
         [4., 4., 4.],
         [4., 4., 4.]]])

In [41]:
z = einsum(x, y, "batch seq1 hidden, batch seq2 hidden -> batch seq1 seq2")
z

tensor([[[4., 4., 4.],
         [4., 4., 4.],
         [4., 4., 4.]],

        [[4., 4., 4.],
         [4., 4., 4.],
         [4., 4., 4.]]])

In [42]:
z = einsum(x, y, "batch seq1 hidden, batch seq2 hidden -> batch seq1 seq2")
z

tensor([[[4., 4., 4.],
         [4., 4., 4.],
         [4., 4., 4.]],

        [[4., 4., 4.],
         [4., 4., 4.],
         [4., 4., 4.]]])

In [43]:
from einops import reduce
x:Float[torch.Tensor, "batch seq hidden"] = torch.ones(2, 3,4)
y = reduce(x, "... hidden -> ...", "mean")
y

tensor([[1., 1., 1.],
        [1., 1., 1.]])

In [44]:
y = x.mean(dim = -1)

In [45]:
y

tensor([[1., 1., 1.],
        [1., 1., 1.]])

In [46]:
from einops import einsum

# 定义两个张量
x: Float[torch.Tensor, "batch seq1 hidden"] = torch.ones(2, 3, 4)
y: Float[torch.Tensor, "batch seq2 hidden"] = torch.ones(2, 3, 4)

# 传统方式
#z = x @ y.transpose(-2, -1)  # batch, sequence, sequence

# Einops方式
z = einsum(x, y, "batch seq1 hidden, batch seq2 hidden -> batch seq1 seq2")
# 未在输出中命名的维度（hidden）会被自动求和
z

tensor([[[4., 4., 4.],
         [4., 4., 4.],
         [4., 4., 4.]],

        [[4., 4., 4.],
         [4., 4., 4.],
         [4., 4., 4.]]])

In [47]:
from einops import rearrange
x: Float[torch.Tensor, "ba se th"] = torch.ones(2, 3,8)
w: Float[torch.Tensor, 'h1 h2'] = torch.ones(4, 4)
x = rearrange(x, "... (heads h1)-> ... heads h1", heads=2)
x


tensor([[[[1., 1., 1., 1.],
          [1., 1., 1., 1.]],

         [[1., 1., 1., 1.],
          [1., 1., 1., 1.]],

         [[1., 1., 1., 1.],
          [1., 1., 1., 1.]]],


        [[[1., 1., 1., 1.],
          [1., 1., 1., 1.]],

         [[1., 1., 1., 1.],
          [1., 1., 1., 1.]],

         [[1., 1., 1., 1.],
          [1., 1., 1., 1.]]]])

In [48]:
x.size()

torch.Size([2, 3, 2, 4])

In [49]:
x = einsum(x, w, "... h1, h1 h2 -> ... h2")

x

tensor([[[[4., 4., 4., 4.],
          [4., 4., 4., 4.]],

         [[4., 4., 4., 4.],
          [4., 4., 4., 4.]],

         [[4., 4., 4., 4.],
          [4., 4., 4., 4.]]],


        [[[4., 4., 4., 4.],
          [4., 4., 4., 4.]],

         [[4., 4., 4., 4.],
          [4., 4., 4., 4.]],

         [[4., 4., 4., 4.],
          [4., 4., 4., 4.]]]])

在 einops 的语法中，空格和括号代表了完全不同的内存布局操作：
1. heads hidden2 (中间有空格) —— 增加维度 (Expand/View)
含义：这是在增加维度的数量。
动作：将一个大的维度“切开”，让它们变成两个独立的坐标轴。
结果：Tensor 的秩（Rank）会增加。比如从 3D 变成 4D。
类比：把一盒 8 颗装的巧克力，摆成 2 行 4 列。
2. (heads hidden2) (带括号) —— 合并维度 (Flatten/Reshape)
含义：这是在减少维度的数量，也叫“扁平化”。
动作：将括号内的多个维度“压扁”成一个维度。
结果：Tensor 的秩会减少。比如从 4D 变回 3D。
类比：把 2 行 4 列摆放的巧克力，重新塞回一个只有 8 个格子的长条形盒子里。


In [50]:
x = rearrange(x, "... heads h2 ->...(heads h2) ")
x

tensor([[[4., 4., 4., 4., 4., 4., 4., 4.],
         [4., 4., 4., 4., 4., 4., 4., 4.],
         [4., 4., 4., 4., 4., 4., 4., 4.]],

        [[4., 4., 4., 4., 4., 4., 4., 4.],
         [4., 4., 4., 4., 4., 4., 4., 4.],
         [4., 4., 4., 4., 4., 4., 4., 4.]]])

In [51]:
x = torch.tensor([
    [0., 1, 2, 3],
    [4, 5, 6, 7],
    [8, 9, 10, 11],
    [12, 13, 14, 15],
])


In [52]:
x

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [12., 13., 14., 15.]])

In [53]:
assert x.stride(0) == 4

In [54]:
assert x.stride(1) == 1

In [55]:
r, c = 1, 2
index  = r * x.stride(0) + c * x.stride(1)
index

6

In [56]:
x = torch.randn(32, 32)
y = x.view(1024)
y.size()

torch.Size([1024])

In [57]:
x.size()

torch.Size([32, 32])

In [58]:
z = x.transpose(0,1)
h = x.transpose(1, 0)
assert torch.equal(z, h)

In [59]:
z.size()

torch.Size([32, 32])

In [60]:
zz = z.view(1024)

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

In [61]:
x = torch.zeros(32, 32)
assert x.device == torch.device("cpu")

In [62]:
if not torch.cuda.is_available():
  print("hello")

In [63]:
import torch
nun_groups = torch.cuda.device_count()
for i in range(nun_groups):
  properties = torch.cuda.get_device_properties(i)

memory_allocated = torch.cuda.memory_allocated()


In [64]:
x = torch.zeros(32, 32)


In [65]:
y = x.to("cuda:0")
assert y.device == torch.device("cuda:0")

In [66]:
z = torch.zeros(32, 32, device="cuda:0")

In [67]:
new_memory_allocated = torch.cuda.memory_allocated()
memory_used = new_memory_allocated - memory_allocated
assert memory_used == 2 * (32 * 32 * 4)

In [68]:
if torch.cuda.is_available():
  B = 16384
  D = 32768
  K = 8192
else:
  B = 1024
  D = 256
  K = 64

device = "cuda" if torch.cuda.is_available() else "cpu"

In [69]:
x = torch.ones(B, D, device=device)
w = torch.randn(D, K, device=device)
y = x @ w
y.size()

torch.Size([16384, 8192])

In [70]:
B

16384

In [71]:
x = torch.tensor([1., 2, 3]) # 输入数据，不需要计算梯度
w = torch.tensor([1., 1, 1], requires_grad=True)

pred_y = x @ w
loss = 0.5 * (pred_y - 5) .pow(2)
loss.backward()

In [72]:
assert loss.grad is None

/tmp/ipykernel_7143/3235844550.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:492.)
  assert loss.grad is None


In [73]:
assert pred_y.grad is None

/tmp/ipykernel_7143/3088157685.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:492.)
  assert pred_y.grad is None


In [74]:
assert x.grad is None

In [75]:
assert torch.equal(w.grad, torch.tensor([1, 2, 3]))

In [76]:
x = torch.ones(B, D, device=device)
w1 = torch.randn(D, D, device=device, requires_grad=True)
w2 = torch.randn(D, D, device=device, requires_grad=True)
h1 = x @ w1
h2 = h1 @ w2
loss = h2.pow(2).mean()


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.91 GiB is free. Including non-PyTorch memory, this process has 12.65 GiB memory in use. Of the allocated memory 12.51 GiB is allocated by PyTorch, and 13.87 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [77]:
import torch.nn as nn

In [79]:
import numpy as np

class Linear(nn.Module):
  def __init__(self, input_dim:int, output_dim:int):
    super().__init__()
    self.weight = nn.Parameter(torch.randn(input_dim, output_dim) / np.sqrt(input_dim))

  def forward(self, x):
    return x @ self.weight


In [80]:
class Cruncher(nn.Module):
  def __init__(self, dim:int, num_layers:int):
    super().__init__()
    self.layers = nn.ModuleList([
        Linear(dim, dim)
        for i in range(num_layers)
    ])
    self.final = Linear(dim, 1)

  def forward(self, x:torch.Tensor) -> torch.Tensor:
    B, D = x.size()
    for layer in self.layers:
      x = layer(x)

    x = self.final(x)
    assert x.size() == torch.Size([B, 1])
    x = x.squeeze(-1)
    assert x.size() == torch.Size([B])
    return x


In [81]:
D = 64
num_layers = 2
nmodel = Cruncher(dim=D, num_layers=num_layers)

In [84]:
device = "cpu"
model = nmodel.to(device)

In [86]:
B = 8 # Batch size
x = torch.randn(B, D, device=device)
y = model(x)
assert y.size() == torch.Size([B])


In [87]:
B

8

In [88]:
orig_data = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=np.int32)

orig_data.tofile("temp.npy")

In [89]:
data = np.memmap("temp.npy", dtype=np.int32)
assert np.array_equal(data, orig_data)

In [ ]:
def get_batch(data: np.array, batch_size: int, sequence_length: int, device: str) -> torch.Tensor:
  start_indices = torch.randint(len(data) - sequence_length, batch_size )
  assert start_indices.size() == torch.Size([batch_size])
  x = torch.tensor([data[start: start + sequence_length] for start in start_indices])
  assert x.size() == torch.Size([batch_size, sequence_length])
  if torch.cuda.is_available():
    x = x.pin_memory()
  x = x.to(device, non_blocking=True)